### RAG Application using typesense

In [2]:
import typesense

In [4]:
### Langchain Typesense Vector Store Example

from langchain_community.document_loaders import TextLoader
from langchain_community.vectorstores import Typesense
from langchain.embeddings import HuggingFaceEmbeddings
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_groq import ChatGroq


In [20]:
import os
from dotenv import load_dotenv
load_dotenv()
groq_api_key = os.getenv("GROQ")
typesense_api_key = os.getenv("TYPESENSE_API_KEY")
typesense_host = os.getenv("TYPE_SCRIPT_NODES")

In [22]:
print(typesense_host)

6btg8eis3jvrofm1p-1.a2.typesense.net


In [11]:
LOADER = TextLoader("../data/AI Blog.txt", encoding="utf-8")
documents = LOADER.load()
text_splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=200)
texts = text_splitter.split_documents(documents)

embeddings = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")


C:\Users\Lenovo\AppData\Local\Temp\ipykernel_7264\1414677975.py:6: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the :class:`~langchain-huggingface package and should be used instead. To use it run `pip install -U :class:`~langchain-huggingface` and import as `from :class:`~langchain_huggingface import HuggingFaceEmbeddings``.
  embeddings = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")
c:\Users\Lenovo\Generative-ai\env\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [23]:
doc_search = Typesense.from_documents(
    texts,
    embeddings,
    typesense_client_params={
        "host": typesense_host,
        "port": "443",
        "protocol": "https",
        "typesense_api_key": typesense_api_key,
        "typesense_collection_name": "lang--chain"
    },
)

In [24]:
query  = "What is the future of AI?"
docs = doc_search.similarity_search(query)
print(docs[0].page_content)

Enterprise leaders have spent the last two years building the case for agentic AI. Proof of concepts. Pilots. Internal demos that impress the room. And a lot of that work has been promising.

But what I keep hearing from customers is that the gap between an AI pilot that works and a process that runs in production is wider than anyone expected. The technology isn't the problem. The challenge is getting AI to operate reliably across the complex, high-stakes processes that actually run a business.

Customers don't just want platforms. They want outcomes. And building an end-to-end agentic process from scratch—even with great tools—takes time, domain expertise, and significant effort to implement, evaluate, and control the results to ensure predictability. The patterns are well known. The processes, while complex and dynamic, are repeatable. What's been missing is a faster, safer path from platform capability to business result.


In [25]:
retriever = doc_search.as_retriever()
retriever

VectorStoreRetriever(tags=['Typesense', 'HuggingFaceEmbeddings'], vectorstore=<langchain_community.vectorstores.typesense.Typesense object at 0x00000253C0B87A30>, search_kwargs={})

In [27]:
retriever.invoke("What is the future of AI?")[0].page_content

"Enterprise leaders have spent the last two years building the case for agentic AI. Proof of concepts. Pilots. Internal demos that impress the room. And a lot of that work has been promising.\n\nBut what I keep hearing from customers is that the gap between an AI pilot that works and a process that runs in production is wider than anyone expected. The technology isn't the problem. The challenge is getting AI to operate reliably across the complex, high-stakes processes that actually run a business.\n\nCustomers don't just want platforms. They want outcomes. And building an end-to-end agentic process from scratch—even with great tools—takes time, domain expertise, and significant effort to implement, evaluate, and control the results to ensure predictability. The patterns are well known. The processes, while complex and dynamic, are repeatable. What's been missing is a faster, safer path from platform capability to business result."